# SOXX 2018-09 原始 N-CSRS 来源

固定 `plan.json` 中唯一 SEC 正文 URL；保存原字节及响应头并完成 Yanxuan NAS 回读。来源身份和数值闭合分步判断。

In [1]:
import json,hashlib,subprocess,tarfile,tempfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_soxx_2018_sep_source_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'attempt_started.json').exists() and sha(root/plan['source_series_calendar'])==plan['source_series_calendar_sha256']
calendar=json.loads((root/plan['source_series_calendar']).read_text());assert any(x['accession']==plan['accession'] and x['report_date']==plan['report_period'] and x['primary_document']==plan['primary_document'] for x in calendar['rows'])
raw=root/'data/snapshots/stage1_soxx_2018_sep_source_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
(card/'attempt_started.json').write_text(json.dumps({'status':'started','body_GET_attempts':1,'market_outcome_reads':0,'model_fits':0,'native_accounts':0},indent=2)+'\n')
body=raw/(plan['accession']+'.html');headers=raw/(plan['accession']+'.headers');started=datetime.now(timezone.utc).isoformat();r=plan['request']
cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(r['connect_timeout_seconds']),'--max-time',str(r['total_timeout_seconds']),'--max-filesize',str(r['max_response_bytes']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',r['user_agent'],'-w','%{http_code} %{url_effective}',plan['url']]
try:
 run=subprocess.run(cmd,capture_output=True,text=True,timeout=210);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
 receipt={'accession':plan['accession'],'report_period':plan['report_period'],'url':plan['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
except Exception as exc:receipt={'accession':plan['accession'],'report_period':plan['report_period'],'url':plan['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
(card/'capture_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'capture_receipt_sha256':sha(card/'capture_receipt.json')};(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-2018-sep-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for x in manifest['files']:out.add(raw/x['name'],arcname='raw/'+x['name'])
  out.add(card/'source_manifest.json',arcname='source_manifest.json');out.add(card/'capture_receipt.json',arcname='capture_receipt.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='soxx-2018-sep-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=150)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for x in manifest['files']:
    member=source.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
result={'status':'one_exact_2018_Sep_SOXX_NCSRS_body_archived_identity_pending' if receipt.get('http_status')==200 and receipt.get('curl_exit_code')==0 and receipt['raw_bytes']<=r['max_response_bytes'] else 'body_transport_needs_repair','receipt':receipt,'nas_archive_sha256':digest,'nas_archive_bytes':evidence['size_bytes'],'nas_remote_and_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'transport_result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','nas_archive_sha256','nas_archive_bytes')})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'one_exact_2018_Sep_SOXX_NCSRS_body_archived_identity_pending', 'nas_archive_sha256': '1d4261348ed2aaceeac1ac1b372c7283061bd785cf3b6ef9fa21ab31a1fc4d14', 'nas_archive_bytes': 2549133}


## 中文结论

2018-09-30 SOXX 原始 N-CSRS 正文 HTTP 200，大小 51,473,821 字节，原文及响应头已 NAS 远端哈希和限定恢复。零网络身份复核发现恰一处与日期和 SOXX 名称连续的 Schedule of Investments / Security / Shares / Value / Common Stocks 表头。0 市场收益、0 拟合、0 账户；普通股数值尚未解析。